# GeoFWI3D — UNETR Dice+CE paired pilot

**One run: seed 0, fold 0.** Test whether adding voxel cross-entropy improves UNETR at the existing cap.
The archived Dice-only control reached **29.12% final validation Dice** (33.04% best).
This is a proposed remedy, not a confirmed fix. No automatic seed sweep follows this pilot.

- Only loss changes: original unsquared Dice including background **+ CE**, weights 1:1.
- Same initialization hash, 19 training / 96 validation volumes, native 96³, batch 1, no augmentation.
- Same AdamW, learning-rate trajectory and BF16 training. No pretrained weights or constraints.
- **182,178.12654316 GFLOPs cap; 414 updates**, with CE included in the accounting.
- Final-budget validation Dice is primary; best validation is secondary. A final score >50% passes
  this single cell only. Three passing seeds across three folds still require separate validation.
- Preflight uses 3 diagnostic updates (~1,318.42 GFLOPs), outside the optimization cap.
  Validation/setup also incur compute. No control rerun is needed.

Sign into **focacciafilippo@gmail.com**, select **A100 GPU + High RAM**, then **Run all**.
The paired comparison requires the control's A100-SXM4-80GB and pinned software; preflight stops if they differ.
Inputs are in `MyDrive/hippo_geofwi3d_unetr_ce_pilot_20261019/input`.
Run one copy only. The notebook verifies initialization before any training update and stops if it differs.
All three folds' existing inputs are included for later reuse; this notebook trains only fold 0.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from googleapiclient.discovery import build
import google.auth
credentials, _ = google.auth.default()
service = build('drive','v3',credentials=credentials,cache_discovery=False)
about = service.about().get(fields='user(emailAddress),storageQuota').execute()
assert about['user']['emailAddress'].lower() == 'focacciafilippo@gmail.com', 'Mount/authenticate the requested account.'
quota=about['storageQuota']
if 'limit' in quota:
    assert int(quota['limit'])-int(quota['usage']) >= 8*1024**3, 'Keep at least 8 GiB free in Drive, including trash.'
from pathlib import Path
import hashlib,json,os,shutil,signal,subprocess,sys,time,zipfile
HOME_DIR=Path('/content/drive/MyDrive/hippo_geofwi3d_unetr_ce_pilot_20261019')
INPUT=HOME_DIR/'input'
RESULTS=HOME_DIR/'results_unetr_dice_ce_f0s0_v1'
BUNDLE=Path('/content/hippo_geofwi_unetr_ce_pilot_v1')
RESULTS.mkdir(parents=True,exist_ok=True)
BUNDLE.mkdir(exist_ok=True)
assert shutil.disk_usage('/content').free >= 20*1024**3, 'Need at least 20 GiB runtime disk space.'
print('Account:',about['user']['emailAddress'],'Results:',RESULTS)


## Verify and stage the uploaded data/code
All archive and per-file SHA256 hashes are checked. The 319 unique images/labels are read from
the runtime SSD during training. Existing matching files are reused; changed files stop execution.


In [ ]:
EXPECTED_UPLOADS={'geofwi3d_unetr_ce_pilot_code.zip': '01e211c83dbc5b30f7ff45540b6aac5fff8bb248ce205dcc2b4ef9e7ed2f0b13', 'geofwi3d_data_part01.zip': '32eaadf1a6abd61dd5bd922a8a085af4e6c876b34b915bbd09da93a3dbea2ca2', 'geofwi3d_data_part02.zip': '31adc6fa680b74a921f9457f24d2dfbb178aa1cef1d80baa152c5ee82a4e05bd', 'geofwi3d_data_part03.zip': 'e9c404631c47a918d1ca36c9166ea92141aa1b8264e332fbaa32b13299d8706a', 'geofwi3d_data_part04.zip': '1ef1e965ebefb571f441d986f2304edb4f2ab824b4bedb7dfbc6a494c4c1c279', 'geofwi3d_data_part05.zip': 'b99cbe88932b94bc382512c059661b48516beacd1dae116237ec411edc64fb58', 'geofwi3d_data_part06.zip': '6f0509326647688300459b05d5f0af5c10ecdde9c19fc6fc3ee05bc92fe8f183', 'geofwi3d_data_part07.zip': '3e1cf8a1cacd445c73bf1798ebd9c1b3c34f6f2cea0e2bf0ad378c6f12f09fb1', 'geofwi3d_data_part08.zip': 'b2b603f5ef55eb2822e64904ce7d178b7cd459c1258398de1f75cdedad5525ed', 'geofwi3d_data_part09.zip': '833fccd13996d6d6ddd118b6146334a902e4eceba2dedd26b798db64fcf6de38', 'geofwi3d_data_part10.zip': '99daf1d8f684ac47897aaa6933b69741e826935783bc42dbd02093efc299e235', 'geofwi3d_data_part11.zip': 'ce12340263414416fcd728e7fad43a2dbbbf4d26f22cc51273acb8df50680239', 'geofwi3d_data_part12.zip': 'ff0f922b1ae82afb2394725bcd0374b4eb273fe0333f5fbad4dbd52b540aadd2', 'geofwi3d_data_part13.zip': '9ede65d6da572c0186b51b838f46f7cd281bbd8b515aef813e502b59891cc880'}
EXPECTED_MANIFEST='592fe8cf4270ae4e031b086a3872abad44f8093c0a6a37c28b937948b1a54ec6'

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()
for name,expected in EXPECTED_UPLOADS.items():
    package=INPUT/name
    if not package.exists():raise FileNotFoundError(str(package))
    assert sha(package)==expected, 'Upload checksum mismatch: '+name
    local=Path('/content')/name
    if not local.exists() or sha(local)!=expected:
        shutil.copyfile(package,local)
    assert sha(local)==expected
    with zipfile.ZipFile(local) as z:
        assert len(z.namelist())==len(set(z.namelist())), 'Duplicate archive members'
        for info in z.infolist():
            p=Path(info.filename)
            assert not p.is_absolute() and '..' not in p.parts
            assert (info.external_attr>>16)&0o170000 != 0o120000
        for info in z.infolist():
            target=BUNDLE/info.filename
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_name(target.name+'.extracting')
            with z.open(info) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst)
            temp.replace(target)
    print('Staged',name)
assert sha(BUNDLE/'bundle_manifest.json')==EXPECTED_MANIFEST
for listing in ['bundle_manifest.json','data_manifest.json']:
    for name,expected in json.loads((BUNDLE/listing).read_text()).items():
        assert sha(BUNDLE/name)==expected, 'Extracted file mismatch: '+name
source=RESULTS/'provenance'/EXPECTED_MANIFEST;source.mkdir(parents=True,exist_ok=True)
for name in ['geofwi3d_unetr_ce_pilot_code.zip']:
    dst=source/name
    if not dst.exists():shutil.copyfile(INPUT/name,dst)
    assert sha(dst)==EXPECTED_UPLOADS[name]
print('All inputs verified. No augmentation.')


## Install the frozen environment
Uses the package versions from the existing Colab training environment, in a separate venv.
Setup output is streamed and saved in Drive. Keep the same GPU type/software on resume.


In [ ]:
VENV=Path('/content/hippo_geofwi_unetr_ce_pilot_venv');PYTHON=VENV/'bin/python'
def command(args,log_name):
    with (RESULTS/log_name).open('a') as log:
        proc=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True)
        tail=[]
        def drain():
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
                tail.append(line);del tail[:-40]
        try:drain();code=proc.wait()
        except KeyboardInterrupt:
            # Signal the orchestrator only; it stops the active trainer at a safe boundary.
            proc.send_signal(signal.SIGINT)
            print('Saving recovery to Drive. Please wait; do not interrupt again.')
            drain();code=proc.wait()
        finally:proc.stdout.close()
        if code:raise RuntimeError('Process failed; recovery retained.\n'+''.join(tail))
if not PYTHON.exists():
    command([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup.log')
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    command([PYTHON,'-m','ensurepip','--upgrade'],'setup.log')
command([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'],'setup.log')
def campaign(action,*extra):
    command([PYTHON,'-u',BUNDLE/'pilot.py',action,'--bundle',BUNDLE,'--durable',RESULTS,*extra],action+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## Required UNETR GPU preflight
Checks one model only: finite loss/gradients, actual 96³ BF16 forward/backward, and exact checkpoint
replay of model, optimizer and RNG. Its 3 diagnostic updates are recorded separately.
The scientific run starts fresh from the matching archived seed-0 initialization.


In [ ]:
campaign('preflight')


## Run or resume the single pilot
The run stops after 414 updates and final evaluation, regardless of its score. It cannot start another
seed/model/fold. Rerunning reuses a verified completed archive or resumes coherent recovery.
Interrupt once and wait for recovery to finish uploading. A hard VM loss can leave charged updates
newer than the checkpoint: the notebook then stops for an audit instead of silently exceeding the cap.


In [ ]:
SESSION_HOURS=2.0
# RESULTS/'STOP' requests a graceful stop; remove only when intentionally resuming.
campaign('run','--hours',str(SESSION_HOURS))


## Compare with the archived Dice-only control
`pilot_status.json` reports the paired difference in percentage points and whether final Dice exceeds
50%. `PILOT_COMPLETE.json` means this one experiment finished, including when the score is below 50%.
It never declares the three-seed/three-fold baseline complete. Curves, confusion counts, config, source,
budget ledger and logs are archived and verified before completed checkpoints are retired.


In [ ]:
campaign('report')
import pandas as pd
status=json.loads((RESULTS/'pilot_status.json').read_text())
print('Pilot status:',status['status'])
rows=[dict(recipe='Archived Dice control',**status['dice_control'])]
rows += [dict(recipe='Dice+CE pilot',**r) for r in status['runs']]
table=pd.DataFrame(rows)
display(table[['recipe','fold','seed','updates','final_dice','best_dice','training_dice']])
if status['runs']:
    print('Final Dice change (percentage points):',status['final_dice_delta_percentage_points'])
    print('Final Dice strictly >50%:',status['threshold_passed'])
    print('Stop here and review before spending more compute.')
else:print('Pilot has not completed. Resume if paused; inspect logs if it failed.')
